# 03 · Silver: clean and validated tables
Splits violations and enforcement actions into separate tables (one row per entity),
converts data types and gives columns readable names.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.sdwa_silver.violations AS
SELECT
  PWSID                                                              AS pwsid,
  VIOLATION_ID                                                       AS violation_id,
  VIOLATION_CODE                                                     AS violation_code,
  VIOLATION_CATEGORY_CODE                                            AS violation_category_code,
  IS_HEALTH_BASED_IND = 'Y'                                          AS is_health_based,
  IS_MAJOR_VIOL_IND = 'Y'                                            AS is_major_violation,
  CONTAMINANT_CODE                                                   AS contaminant_code,
  RULE_CODE                                                          AS rule_code,
  RULE_FAMILY_CODE                                                   AS rule_family_code,
  VIOLATION_STATUS                                                   AS violation_status,
  to_date(try_to_timestamp(NON_COMPL_PER_BEGIN_DATE, 'MM/dd/yyyy'))  AS noncompliance_begin_date,
  to_date(try_to_timestamp(NON_COMPL_PER_END_DATE, 'MM/dd/yyyy'))    AS noncompliance_end_date,
  to_date(try_to_timestamp(CALCULATED_RTC_DATE, 'MM/dd/yyyy'))       AS returned_to_compliance_date,
  _release
FROM workspace.sdwa_bronze.violations_enforcement
WHERE VIOLATION_ID IS NOT NULL
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY PWSID, VIOLATION_ID
  ORDER BY try_to_timestamp(VIOL_LAST_REPORTED_DATE, 'MM/dd/yyyy') DESC
) = 1

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.sdwa_silver.enforcement_actions AS
SELECT DISTINCT
  PWSID                                                      AS pwsid,
  VIOLATION_ID                                               AS violation_id,
  ENFORCEMENT_ID                                             AS enforcement_id,
  to_date(try_to_timestamp(ENFORCEMENT_DATE, 'MM/dd/yyyy'))  AS enforcement_date,
  ENFORCEMENT_ACTION_TYPE_CODE                               AS enforcement_action_type_code,
  ENF_ACTION_CATEGORY                                        AS enforcement_action_category,
  _release
FROM workspace.sdwa_bronze.violations_enforcement
WHERE ENFORCEMENT_ID IS NOT NULL

In [0]:
%sql
SELECT
  COUNT(*)                                                AS violations,
  COUNT_IF(noncompliance_begin_date IS NULL)              AS missing_begin_date,
  COUNT_IF(is_health_based)                               AS health_based,
  (SELECT COUNT(*) FROM workspace.sdwa_silver.enforcement_actions) AS enforcement_actions
FROM workspace.sdwa_silver.violations